# LeetCode 1986 — Minimum Number of Work Sessions to Finish the Tasks bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-work-sessions-to-finish-the-tasks/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Backtracking, Bit Manipulation, Bitmask

Skeleton LeetCode:

```python
class Solution:
    def minSessions(self, tasks: List[int], sessionTime: int) -> int:
        ...
```

Đây là bài **xếp thùng (bin packing)** cỡ nhỏ (`n ≤ 14`). Bin packing tổng quát là NP-khó, nên lời giải dựa vào `n` nhỏ: **vét cạn mọi cách chia** → **quay lui + cắt tỉa** → **DP trên tập con `O(3ⁿ)`** → **DP bitmask lưu cặp (số phiên, tải phiên cuối) `O(n·2ⁿ)`**.

## 1. Mục tiêu học

1. Nhận ra mô hình **bin packing**: mỗi phiên làm việc là một "thùng" sức chứa `sessionTime`.
2. Hiểu vì sao **tham lam** (First-Fit Decreasing…) **không** tối ưu — có phản ví dụ cụ thể.
3. Viết **DP trên tập con**: `f[mask] = min_{sub ⊆ mask, sum(sub) ≤ T} f[mask \ sub] + 1`, duyệt submask bằng `sub = (sub − 1) & mask` — tổng `O(3ⁿ)`.
4. Viết **DP bitmask lưu cặp** `(số phiên, tải phiên đang mở)` và chứng minh vì sao so sánh theo thứ tự từ điển là đủ.
5. Viết **quay lui có cắt tỉa** (sắp giảm dần, bỏ thùng trùng tải, chặn theo kết quả tốt nhất).

## 2. Đề bài & ví dụ

Có `n` công việc, công việc `i` mất `tasks[i]` giờ. Một **phiên làm việc** kéo dài **tối đa** `sessionTime` giờ liên tục rồi nghỉ.

- Bắt đầu công việc nào trong phiên nào thì phải làm xong trong **chính phiên đó**.
- Có thể bắt đầu việc mới ngay sau khi xong việc trước.
- Làm theo **thứ tự bất kỳ**.

Trả về **số phiên tối thiểu** để làm xong tất cả. Đảm bảo `sessionTime ≥ max(tasks)`.

**Ví dụ 1:** `tasks = [1,2,3]`, `sessionTime = 3` → `2` (phiên 1: `1+2`, phiên 2: `3`).

**Ví dụ 2:** `tasks = [3,1,3,1,1]`, `sessionTime = 8` → `2` (`3+1+3+1 = 8`, rồi `1`).

**Ví dụ 3:** `tasks = [1,2,3,4,5]`, `sessionTime = 15` → `1`.

**Ràng buộc:** `n == len(tasks)`, `1 ≤ n ≤ 14`, `1 ≤ tasks[i] ≤ 10`, `max(tasks) ≤ sessionTime ≤ 15`.

In [1]:
import random
import time
from functools import lru_cache
from typing import List

examples = [([1, 2, 3], 3, 2), ([3, 1, 3, 1, 1], 8, 2), ([1, 2, 3, 4, 5], 15, 1)]
for tasks, T, exp in examples:
    print(f"tasks={tasks}, T={T}: tổng={sum(tasks)}, cận dưới ceil(tổng/T)={-(-sum(tasks) // T)}, đáp án={exp}")

tasks=[1, 2, 3], T=3: tổng=6, cận dưới ceil(tổng/T)=2, đáp án=2
tasks=[3, 1, 3, 1, 1], T=8: tổng=9, cận dưới ceil(tổng/T)=2, đáp án=2
tasks=[1, 2, 3, 4, 5], T=15: tổng=15, cận dưới ceil(tổng/T)=1, đáp án=1


## 3. Kiến thức nền

### 3.1 Bitmask biểu diễn tập công việc

Bit `i` của `mask` bằng 1 ⇔ công việc `i` thuộc tập. `n ≤ 14` ⇒ `2¹⁴ = 16384` tập.

In [2]:
tasks = [1, 2, 3]
for mask in range(1 << len(tasks)):
    chosen = [tasks[i] for i in range(len(tasks)) if mask >> i & 1]
    print(f"mask={mask:03b} -> {chosen}, tổng {sum(chosen)}")

mask=000 -> [], tổng 0
mask=001 -> [1], tổng 1
mask=010 -> [2], tổng 2
mask=011 -> [1, 2], tổng 3
mask=100 -> [3], tổng 3
mask=101 -> [1, 3], tổng 4
mask=110 -> [2, 3], tổng 5
mask=111 -> [1, 2, 3], tổng 6


### 3.2 Tổng theo mask trong `O(2ⁿ)`

`s[mask] = s[mask ^ lowbit] + tasks[index(lowbit)]` với `lowbit = mask & -mask`.

In [3]:
def subset_sums(tasks):
    n = len(tasks)
    s = [0] * (1 << n)
    for m in range(1, 1 << n):
        lb = m & -m
        s[m] = s[m ^ lb] + tasks[lb.bit_length() - 1]
    return s

print(subset_sums([1, 2, 3]))

[0, 1, 2, 3, 3, 4, 5, 6]


### 3.3 Duyệt mọi tập con khác rỗng của `mask`

```python
sub = mask
while sub:
    ...
    sub = (sub - 1) & mask
```

Tổng số cặp `(mask, sub ⊆ mask)` là `3ⁿ` (mỗi phần tử: ngoài mask / trong mask ngoài sub / trong sub).

In [4]:
mask = 0b1011
subs = []
sub = mask
while sub:
    subs.append(format(sub, '04b'))
    sub = (sub - 1) & mask
print("các tập con khác rỗng của 1011:", subs)
print("số cặp (mask, sub) với n=14:", 3 ** 14, "| so với 4^14 =", 4 ** 14)

các tập con khác rỗng của 1011: ['1011', '1010', '1001', '1000', '0011', '0010', '0001']
số cặp (mask, sub) với n=14: 4782969 | so với 4^14 = 268435456


### 3.4 Bin packing và cận dưới

Cận dưới hiển nhiên: `ceil(sum / T)`. Không phải lúc nào cũng đạt: `[5,5,5]`, `T = 8` cần 3 phiên dù `ceil(15/8) = 2` — không hai việc nào ghép chung được.

## 4. Ý tưởng nền tảng / trực giác

Thứ tự làm **trong** một phiên không quan trọng; chỉ quan trọng **tập** việc trong mỗi phiên có tổng `≤ T`. ⇒ Bài toán = **chia tập công việc thành ít nhóm nhất**, mỗi nhóm tổng `≤ T`.

Ví dụ 2: `[3,1,3,1,1]`, `T = 8`: tổng 9 > 8 nên ≥ 2 phiên; nhóm `{3,1,3,1}` và `{1}` ⇒ đúng 2.

Với `n ≤ 14`, ta có thể duyệt **tập con** (16384 trạng thái) — gợi ý mạnh cho DP bitmask.

In [5]:
def show_partition(groups, T):
    for g in groups:
        print(f"  phiên {g} tổng {sum(g)} {'≤' if sum(g) <= T else '>'} {T}")

show_partition([[3, 1, 3, 1], [1]], 8)

  phiên [3, 1, 3, 1] tổng 8 ≤ 8
  phiên [1] tổng 1 ≤ 8


## 5. Cách 1 — Brute Force / baseline: vét cạn mọi cách chia nhóm

Lần lượt đặt từng việc vào **một phiên đã mở** (nếu còn chỗ) hoặc **mở phiên mới**. Duyệt hết mọi khả năng, lấy số phiên nhỏ nhất. Số cách chia tập `n` phần tử là số Bell `B(n)` — `B(14) ≈ 1.9·10⁸`: quá lớn cho `n = 14`, nhưng dùng làm **chuẩn đối chiếu** với `n ≤ 8`.

In [6]:
def min_sessions_brute(tasks, T):
    best = [len(tasks)]
    loads = []
    def go(i):
        if i == len(tasks):
            best[0] = min(best[0], len(loads)); return
        for j in range(len(loads)):
            if loads[j] + tasks[i] <= T:
                loads[j] += tasks[i]; go(i + 1); loads[j] -= tasks[i]
        loads.append(tasks[i]); go(i + 1); loads.pop()
    go(0)
    return best[0]

for tasks_, T_, exp in examples:
    print(min_sessions_brute(tasks_, T_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(B(n) · n)` — bùng nổ khi `n` lớn. **Nút thắt:** cùng một **tập** việc đã xếp được xét lại nhiều lần theo nhiều cách chia khác nhau ⇒ nhớ theo **tập** (bitmask).

## 6. Ý tưởng thuật toán / bước đột phá

**DP trên tập con.** `f[mask]` = số phiên tối thiểu để làm xong các việc trong `mask`. Phiên **cuối cùng** là một tập con `sub ⊆ mask` có tổng `≤ T`:

`f[0] = 0`, `f[mask] = min over sub ⊆ mask, sub ≠ ∅, s[sub] ≤ T của f[mask ^ sub] + 1`.

**DP cặp, nhanh hơn.** Xếp việc theo **một thứ tự** nào đó, các phiên lần lượt được mở. Với tập `mask` đã xếp, lưu `dp[mask] = (số phiên đã mở, tải phiên đang mở)` **nhỏ nhất theo thứ tự từ điển**. Thêm việc `i`: **nếu** tải + `tasks[i] ≤ T` **thì** bỏ vào phiên đang mở, **ngược lại** mở phiên mới với tải `tasks[i]`. Độ phức tạp `O(n · 2ⁿ)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quay lui + cắt tỉa (branch and bound)

Cải tiến cách 1:
- **Sắp giảm dần**: việc lớn khó xếp, đặt trước ⇒ nhánh xấu chết sớm.
- **Chặn**: nếu số phiên đang mở `≥ best` thì cắt.
- **Bỏ phiên trùng tải**: hai phiên cùng tải cho kết quả đối xứng — chỉ thử một.

In [7]:
def min_sessions_bt(tasks, T):
    a = sorted(tasks, reverse=True)
    best = [len(a)]
    loads = []
    def go(i):
        if len(loads) >= best[0]:
            return                                   # chặn: không thể tốt hơn
        if i == len(a):
            best[0] = len(loads); return
        tried = set()
        for j in range(len(loads)):
            if loads[j] + a[i] <= T and loads[j] not in tried:
                tried.add(loads[j])                  # phiên cùng tải -> đối xứng
                loads[j] += a[i]; go(i + 1); loads[j] -= a[i]
        loads.append(a[i]); go(i + 1); loads.pop()
    go(0)
    return best[0]

for tasks_, T_, exp in examples:
    print(min_sessions_bt(tasks_, T_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )


**Độ phức tạp:** xấu nhất vẫn mũ, nhưng với `n ≤ 14` cắt tỉa giúp chạy rất nhanh trên thực tế. Lưu ý: điều kiện `len(loads) >= best` đặt **đầu** hàm vẫn đúng vì khi `i == len(a)` ta chỉ cập nhật nếu `len(loads) < best`.

### 7.2 Cách 3 — DP trên tập con `O(3ⁿ)`

In [8]:
def min_sessions_subset(tasks, T):
    n = len(tasks)
    s = subset_sums(tasks)
    INF = n + 1
    f = [INF] * (1 << n)
    f[0] = 0
    for mask in range(1, 1 << n):
        sub = mask
        best = INF
        while sub:
            if s[sub] <= T:
                v = f[mask ^ sub] + 1
                if v < best:
                    best = v
            sub = (sub - 1) & mask
        f[mask] = best
    return f[(1 << n) - 1]

for tasks_, T_, exp in examples:
    print(min_sessions_subset(tasks_, T_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(3ⁿ)` ≈ `4.8·10⁶` bước với `n = 14` — từ vài trăm ms đến vài giây trong Python tùy máy (xem đo ở mục 11). Bộ nhớ `O(2ⁿ)`.

**Tối ưu nhỏ:** chỉ xét `sub` **chứa bit thấp nhất của `mask`** (phiên chứa việc có chỉ số nhỏ nhất) — tránh đếm cùng một cách chia theo nhiều thứ tự phiên, giảm số bước đáng kể.

In [9]:
def min_sessions_subset_lowbit(tasks, T):
    n = len(tasks)
    s = subset_sums(tasks)
    f = [n + 1] * (1 << n)
    f[0] = 0
    for mask in range(1, 1 << n):
        low = mask & -mask
        rest = mask ^ low
        sub = rest
        best = n + 1
        while True:
            cur = sub | low                            # luôn chứa bit thấp nhất
            if s[cur] <= T and f[mask ^ cur] + 1 < best:
                best = f[mask ^ cur] + 1
            if sub == 0:
                break
            sub = (sub - 1) & rest
        f[mask] = best
    return f[(1 << n) - 1]

for tasks_, T_, exp in examples:
    print(min_sessions_subset_lowbit(tasks_, T_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )


### 7.3 Cách 4 — DP bitmask lưu cặp `(số phiên, tải phiên cuối)` `O(n·2ⁿ)`

In [10]:
def min_sessions_pair(tasks, T):
    n = len(tasks)
    INF = (n + 1, 0)
    dp = [INF] * (1 << n)
    dp[0] = (1, 0)                             # một phiên đang mở, tải 0
    for mask in range(1 << n):
        cnt, load = dp[mask]
        if cnt > n:
            continue
        for i in range(n):
            if mask >> i & 1:
                continue
            t = tasks[i]
            cand = (cnt, load + t) if load + t <= T else (cnt + 1, t)
            nm = mask | 1 << i
            if cand < dp[nm]:
                dp[nm] = cand
    return dp[(1 << n) - 1][0]

for tasks_, T_, exp in examples:
    print(min_sessions_pair(tasks_, T_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n · 2ⁿ)` ≈ `2.3·10⁵` bước với `n = 14`. Bộ nhớ `O(2ⁿ)`. Đây là cách nộp bài.

**Mẹo:** `n = 1` thì đáp án 1 (vì `T ≥ max`). Với `n = 0` (không có trong đề) cách này trả 1 — không cần xử lý.

### 7.4 Cách 5 — Thử dần số phiên `k` + kiểm tra khả thi bằng quay lui

Bắt đầu từ cận dưới `k = ceil(sum/T)`, tăng dần; với mỗi `k` hỏi "chia được vào `k` thùng sức chứa `T` không?" bằng quay lui có cắt tỉa. Tư duy này giống LeetCode 698 (Partition to K Equal Sum Subsets).

In [11]:
def can_fit(a, T, k):
    bins = [0] * k
    def go(i):
        if i == len(a):
            return True
        seen = set()
        for j in range(k):
            if bins[j] + a[i] <= T and bins[j] not in seen:
                seen.add(bins[j])
                bins[j] += a[i]
                if go(i + 1):
                    return True
                bins[j] -= a[i]
        return False
    return go(0)

def min_sessions_iter_k(tasks, T):
    a = sorted(tasks, reverse=True)
    k = max(1, -(-sum(a) // T))
    while not can_fit(a, T, k):
        k += 1
    return k

for tasks_, T_, exp in examples:
    print(min_sessions_iter_k(tasks_, T_), "(kỳ vọng", exp, ")")

2 (kỳ vọng 2 )
2 (kỳ vọng 2 )
1 (kỳ vọng 1 )


**Độ phức tạp:** mũ trong xấu nhất; thường chỉ thử 1–2 giá trị `k` vì đáp án hay sát cận dưới.

## 8. TRACE từng bước

### 8.1 Trace DP trên tập con với ví dụ 1 `[1,2,3]`, `T = 3`

In [12]:
def trace_subset(tasks, T):
    n = len(tasks)
    s = subset_sums(tasks)
    f = [n + 1] * (1 << n); f[0] = 0
    for mask in range(1, 1 << n):
        sub = mask; opts = []
        while sub:
            if s[sub] <= T:
                opts.append((f[mask ^ sub] + 1, sub))
            sub = (sub - 1) & mask
        f[mask] = min(opts)[0]
        v, sb = min(opts)
        items = [tasks[i] for i in range(n) if mask >> i & 1]
        last = [tasks[i] for i in range(n) if sb >> i & 1]
        print(f"  mask={mask:0{n}b} {items}: phiên cuối tốt nhất {last} -> f = f[{mask ^ sb:0{n}b}] + 1 = {v}")
    return f[-1]

print("=>", trace_subset([1, 2, 3], 3))

  mask=001 [1]: phiên cuối tốt nhất [1] -> f = f[000] + 1 = 1
  mask=010 [2]: phiên cuối tốt nhất [2] -> f = f[000] + 1 = 1
  mask=011 [1, 2]: phiên cuối tốt nhất [1, 2] -> f = f[000] + 1 = 1
  mask=100 [3]: phiên cuối tốt nhất [3] -> f = f[000] + 1 = 1
  mask=101 [1, 3]: phiên cuối tốt nhất [1] -> f = f[100] + 1 = 2
  mask=110 [2, 3]: phiên cuối tốt nhất [2] -> f = f[100] + 1 = 2
  mask=111 [1, 2, 3]: phiên cuối tốt nhất [1, 2] -> f = f[100] + 1 = 2
=> 2


### 8.2 Trace DP cặp với ví dụ 2 `[3,1,3,1,1]`, `T = 8` (chỉ in các mask theo một đường tối ưu)

In [13]:
def trace_pair(tasks, T):
    n = len(tasks)
    dp = [(n + 1, 0)] * (1 << n); dp[0] = (1, 0)
    parent = [None] * (1 << n)
    for mask in range(1 << n):
        cnt, load = dp[mask]
        if cnt > n:
            continue
        for i in range(n):
            if mask >> i & 1:
                continue
            t = tasks[i]
            cand = (cnt, load + t) if load + t <= T else (cnt + 1, t)
            nm = mask | 1 << i
            if cand < dp[nm]:
                dp[nm] = cand; parent[nm] = (mask, i)
    path = []
    m = (1 << n) - 1
    while m:
        pm, i = parent[m]; path.append((m, i)); m = pm
    for m, i in reversed(path):
        print(f"  thêm việc #{i} (={tasks[i]}): mask={m:0{n}b}, (số phiên, tải phiên đang mở) = {dp[m]}")
    return dp[-1][0]

print("=>", trace_pair([3, 1, 3, 1, 1], 8))

  thêm việc #0 (=3): mask=00001, (số phiên, tải phiên đang mở) = (1, 3)
  thêm việc #1 (=1): mask=00011, (số phiên, tải phiên đang mở) = (1, 4)
  thêm việc #2 (=3): mask=00111, (số phiên, tải phiên đang mở) = (1, 7)
  thêm việc #3 (=1): mask=01111, (số phiên, tải phiên đang mở) = (1, 8)
  thêm việc #4 (=1): mask=11111, (số phiên, tải phiên đang mở) = (2, 1)
=> 2


### 8.3 Trace quay lui trên `[5,3,6,5,7,4]`, `T = 15` (ví dụ tự dựng, đã đối chiếu vét cạn)

In [14]:
def trace_bt(tasks, T):
    a = sorted(tasks, reverse=True)
    best = [len(a)]; loads = []; log = []
    def go(i, depth=0):
        if len(loads) >= best[0]:
            return
        if i == len(a):
            best[0] = len(loads); log.append(f"  tìm được {len(loads)} phiên: tải {list(loads)}"); return
        tried = set()
        for j in range(len(loads)):
            if loads[j] + a[i] <= T and loads[j] not in tried:
                tried.add(loads[j]); loads[j] += a[i]; go(i + 1, depth + 1); loads[j] -= a[i]
        loads.append(a[i]); go(i + 1, depth + 1); loads.pop()
    go(0)
    print("\n".join(log))
    return best[0]

print("=>", trace_bt([5, 3, 6, 5, 7, 4], 15), "| vét cạn:", min_sessions_brute([5, 3, 6, 5, 7, 4], 15))

  tìm được 3 phiên: tải [13, 14, 3]
  tìm được 2 phiên: tải [15, 15]
=> 2 | vét cạn: 2


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def minSessions(self, tasks: List[int], sessionTime: int) -> int:
        n = len(tasks)
        full = (1 << n) - 1
        dp = [(n + 1, 0)] * (1 << n)
        dp[0] = (1, 0)                          # (số phiên đã mở, tải của phiên đang mở)
        for mask in range(full + 1):
            cnt, load = dp[mask]
            if cnt > n:
                continue
            for i in range(n):
                if mask >> i & 1:
                    continue
                t = tasks[i]
                cand = (cnt, load + t) if load + t <= sessionTime else (cnt + 1, t)
                nm = mask | (1 << i)
                if cand < dp[nm]:
                    dp[nm] = cand
        return dp[full][0]

for tasks_, T_, exp in examples:
    assert Solution().minSessions(tasks_, T_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**DP trên tập con.** Trong mọi lời giải tối ưu cho `mask`, bỏ đi một phiên `sub` (tổng `≤ T`) thì phần còn lại là một lời giải cho `mask ^ sub` dùng ít hơn 1 phiên ⇒ `f[mask] ≥ f[mask ^ sub] + 1 ≥ min(...)`. Ngược lại, mọi lựa chọn `sub` hợp lệ cộng với lời giải tối ưu của `mask ^ sub` là một lời giải hợp lệ ⇒ `f[mask] ≤ min(...)`.

**DP cặp.** Mọi lời giải có `m` phiên có thể "làm phẳng" thành một **thứ tự** việc: hết phiên 1 rồi phiên 2… Khi đi theo thứ tự đó, quy tắc "vừa thì cho vào phiên đang mở, không vừa thì mở phiên mới" dùng **không quá** `m` phiên (mỗi lần mở phiên mới chỉ xảy ra không muộn hơn lúc lời giải gốc chuyển phiên). Vậy tối ưu = min trên mọi thứ tự.

Còn phải chứng minh giữ **cặp nhỏ nhất theo từ điển** cho mỗi `mask` là đủ: nếu `(c₁, l₁) < (c₂, l₂)` thì
- `c₁ < c₂`: trạng thái 1 có thể "giả vờ" mở phiên mới ngay (tải 0) và vẫn có `c₁ + 1 ≤ c₂` phiên với tải `0 ≤ l₂` — không tệ hơn.
- `c₁ = c₂`, `l₁ ≤ l₂`: tải nhỏ hơn thì mọi việc vừa với trạng thái 2 cũng vừa với trạng thái 1.

Nên trạng thái từ điển nhỏ hơn **trội** hơn (dominates) ⇒ chỉ cần giữ nó.

**Quay lui.** Duyệt mọi phân hoạch (trừ các nhánh đối xứng hoặc chắc chắn không tốt hơn `best`) ⇒ tìm được tối ưu.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn phân hoạch | `O(B(n) · n)` | `O(n)` |
| 2. Quay lui + cắt tỉa | mũ (thực tế nhanh) | `O(n)` |
| 3. DP tập con | `O(3ⁿ)` (bản lowbit: `≈ 3ⁿ/2`) | `O(2ⁿ)` |
| 4. DP cặp | `O(n · 2ⁿ)` | `O(2ⁿ)` |
| 5. Thử dần `k` | mũ (thực tế nhanh) | `O(n)` |

In [16]:
rnd = random.Random(1986)
worst = [rnd.randint(1, 10) for _ in range(14)]
for name, fn in [("bt", min_sessions_bt), ("subset_lowbit", min_sessions_subset_lowbit), ("pair", min_sessions_pair), ("iter_k", min_sessions_iter_k), ("Solution", Solution().minSessions)]:
    t0 = time.perf_counter(); r = fn(worst, 15); dt = time.perf_counter() - t0
    print(f"{name:<14}: {r}  {dt * 1000:8.1f} ms")
t0 = time.perf_counter(); r = min_sessions_subset(worst, 15); dt = time.perf_counter() - t0
print(f"{'subset (3^n)':<14}: {r}  {dt * 1000:8.1f} ms")

bt            : 5       0.0 ms


subset_lowbit : 5     149.7 ms


pair          : 5      21.2 ms
iter_k        : 5       0.0 ms


Solution      : 5      20.3 ms

subset (3^n)  : 5     225.2 ms


## 12. Bộ test

In [17]:
ALL = {
    "bt": min_sessions_bt,
    "subset": min_sessions_subset,
    "subset_lowbit": min_sessions_subset_lowbit,
    "pair": min_sessions_pair,
    "iter_k": min_sessions_iter_k,
    "Solution": lambda a, T: Solution().minSessions(a, T),
}

def check(tasks, T, exp=None, brute=True):
    assert max(tasks) <= T
    ref = min_sessions_brute(tasks, T) if brute else min_sessions_pair(tasks, T)
    for name, fn in ALL.items():
        if name == "subset" and len(tasks) > 11:
            continue                                   # 3^n chậm, chỉ chạy cỡ vừa
        got = fn(list(tasks), T)
        assert got == ref, (name, tasks, T, got, ref)
    if exp is not None:
        assert ref == exp, (tasks, T, ref, exp)
    return ref

for tasks_, T_, exp in examples:
    check(tasks_, T_, exp)
print("Ví dụ đề: OK")

check([5], 5, 1)                                # một việc
check([5, 5, 5], 8, 3)                          # không ghép được
check([1] * 14, 15, 1, brute=False)             # vừa hết một phiên
check([10] * 14, 10, 14, brute=False)           # mỗi việc một phiên
check([5, 3, 6, 5, 7, 4], 15, 2)                # tham lam sai (mục 13), tối ưu 2 — đối chiếu vét cạn
check([3, 4, 5, 5, 4], 11, 2)
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(400):
    n = rnd.randint(1, 8); T = rnd.randint(1, 15)
    tasks_ = [rnd.randint(1, min(10, T)) for _ in range(n)]
    check(tasks_, T)
print("400 test ngẫu nhiên n ≤ 8 so với vét cạn: OK")

for _ in range(15):
    n = rnd.randint(9, 14); T = rnd.randint(10, 15)
    tasks_ = [rnd.randint(1, 10) for _ in range(n)]
    check(tasks_, T, brute=False)
print("15 test n ∈ [9, 14] (đối chiếu chéo các cách): OK")

Ví dụ đề: OK


Edge case: OK
400 test ngẫu nhiên n ≤ 8 so với vét cạn: OK


15 test n ∈ [9, 14] (đối chiếu chéo các cách): OK


In [18]:
# Kiểm cận dưới / cận trên: ceil(sum/T) ≤ đáp án ≤ số phiên của First-Fit Decreasing
def ffd(tasks, T):
    bins = []
    for t in sorted(tasks, reverse=True):
        for j in range(len(bins)):
            if bins[j] + t <= T:
                bins[j] += t; break
        else:
            bins.append(t)
    return len(bins)

rnd = random.Random(7)
gap = 0
for _ in range(500):
    n = rnd.randint(1, 10); T = rnd.randint(1, 15)
    a = [rnd.randint(1, min(10, T)) for _ in range(n)]
    ans = min_sessions_pair(a, T)
    assert -(-sum(a) // T) <= ans <= ffd(a, T)
    gap += ffd(a, T) != ans
print("cận đúng trên 500 test; số lần FFD không tối ưu:", gap)

cận đúng trên 500 test; số lần FFD không tối ưu: 0


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam First-Fit Decreasing (sắp giảm rồi bỏ vào phiên đầu tiên còn vừa)

In [19]:
tasks_, T_ = [5, 3, 6, 5, 7, 4], 15
print("FFD (sai) :", ffd(tasks_, T_))
print("đúng      :", Solution().minSessions(tasks_, T_), "— ví dụ {7,5,3} và {6,5,4}, mỗi phiên đúng 15")

FFD (sai) : 3
đúng      : 2 — ví dụ {7,5,3} và {6,5,4}, mỗi phiên đúng 15


**Lưu ý:** ở mục 12, FFD cho đúng trên cả 500 test ngẫu nhiên — tham lam bin packing **hiếm khi** sai trên dữ liệu ngẫu nhiên nhỏ, nên test ngẫu nhiên dễ "lọt" lỗi; phản ví dụ dưới đây được tìm bằng cách quét có chủ đích hàng chục nghìn bộ.

**Vì sao sai:** FFD đặt `7, 6` chung (13), rồi `5, 5, 4` (14), còn `3` không vừa đâu ⇒ 3 phiên. Bin packing là NP-khó, không có tham lam tối ưu. **Sửa:** DP bitmask.

### Lỗi 2 — Next-Fit theo thứ tự đầu vào (chỉ giữ một phiên mở)

In [20]:
def next_fit(tasks, T):
    c, load = 1, 0
    for t in tasks:
        if load + t <= T:
            load += t
        else:
            c += 1; load = t
    return c

tasks_, T_ = [5, 11, 5], 11
print("next-fit (sai):", next_fit(tasks_, T_), "| đúng:", Solution().minSessions(tasks_, T_))

next-fit (sai): 3 | đúng: 2


**Vì sao sai:** đề cho phép **thứ tự bất kỳ**; `5` và `5` ghép được nếu đổi thứ tự. DP cặp đúng là vì nó lấy **min trên mọi thứ tự** (mọi `mask`), không chỉ thứ tự đầu vào.

### Lỗi 3 — DP cặp: mở phiên mới nhưng đặt tải về 0

In [21]:
def pair_bug_reset(tasks, T):
    n = len(tasks)
    dp = [(n + 1, 0)] * (1 << n); dp[0] = (1, 0)
    for mask in range(1 << n):
        cnt, load = dp[mask]
        for i in range(n):
            if not mask >> i & 1:
                t = tasks[i]
                cand = (cnt, load + t) if load + t <= T else (cnt + 1, 0)   # SAI: tải phải là t
                nm = mask | 1 << i
                dp[nm] = min(dp[nm], cand)
    return dp[-1][0]

tasks_, T_ = [9, 4, 7], 9
print("sai :", pair_bug_reset(tasks_, T_), "| đúng:", Solution().minSessions(tasks_, T_))

sai : 2 | đúng: 3


**Vì sao sai:** việc vừa làm "biến mất" khỏi phiên mới, nên `4` và `7` được ghép sai. **Sửa:** `(cnt + 1, t)`.

### Lỗi 4 — DP cặp chỉ so sánh số phiên, bỏ qua tải

In [22]:
def pair_bug_count_only(tasks, T):
    n = len(tasks)
    cnt = [n + 1] * (1 << n); load = [0] * (1 << n); cnt[0] = 1
    for mask in range(1 << n):
        for i in range(n):
            if not mask >> i & 1:
                nm = mask | 1 << i
                if load[mask] + tasks[i] <= T:
                    c, l = cnt[mask], load[mask] + tasks[i]
                else:
                    c, l = cnt[mask] + 1, tasks[i]
                if c < cnt[nm]:                     # SAI: cùng số phiên mà tải nhỏ hơn thì không cập nhật
                    cnt[nm], load[nm] = c, l
    return cnt[-1]

tasks_, T_ = [3, 4, 5, 5, 4], 11
print("sai :", pair_bug_count_only(tasks_, T_), "| đúng:", Solution().minSessions(tasks_, T_))

sai : 3 | đúng: 2


**Vì sao sai:** hai trạng thái cùng số phiên nhưng tải khác nhau không tương đương; giữ tải lớn hơn làm mất cơ hội ghép. **Sửa:** so sánh **cả cặp** `(cnt, load)` theo thứ tự từ điển (tuple Python làm sẵn).

### Lỗi 5 — Trả về cận dưới `ceil(sum / T)`

In [23]:
def lower_bound_only(tasks, T):
    return -(-sum(tasks) // T)

tasks_, T_ = [5, 5, 5], 8
print("ceil(sum/T) (sai):", lower_bound_only(tasks_, T_), "| đúng:", Solution().minSessions(tasks_, T_))

ceil(sum/T) (sai): 2 | đúng: 3


**Sửa:** cận dưới chỉ dùng để **cắt tỉa** hoặc làm điểm xuất phát (cách 5), không phải đáp án.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n ≤ 14..16` + "chia thành nhóm" ⇒ nghĩ ngay **bitmask DP**.
- Nhóm có **sức chứa** và muốn **ít nhóm nhất** ⇒ DP cặp `(số nhóm, tải nhóm cuối)` `O(n·2ⁿ)`.
- Muốn tối ưu một **hàm của từng nhóm** (ví dụ max, tổng chi phí) ⇒ DP trên tập con `O(3ⁿ)`.
- Tuple Python so sánh từ điển — tiện cho trạng thái `(cnt, load)`.
- Quay lui: **sắp giảm dần** + **bỏ thùng trùng tải** + **chặn theo best**.

## 15. Biến thể của bài

1. **LeetCode 1723 — Find Minimum Time to Finish All Jobs:** cố định `k` người, tối thiểu hóa tải lớn nhất ⇒ DP tập con `f[j][mask]` hoặc nhị phân + quay lui.
2. **LeetCode 2305 — Fair Distribution of Cookies:** giống 1723 với `n ≤ 8`.
3. **LeetCode 698 — Partition to K Equal Sum Subsets:** hỏi khả thi chia `k` nhóm tổng bằng nhau.
4. **LeetCode 1049 / 2035** (chia hai nhóm gần bằng nhau): knapsack / meet-in-the-middle.
5. **Giới hạn số việc mỗi phiên** (tối đa `c` việc): thêm điều kiện `popcount(sub) ≤ c` trong DP tập con (bài tập 3).

In [24]:
def min_max_load_k_workers(jobs, k):
    """LeetCode 1723: chia jobs cho k người, tối thiểu hóa tải lớn nhất — DP tập con."""
    n = len(jobs)
    s = subset_sums(jobs)
    f = s[:]                                # 1 người: tải = tổng mask
    for _ in range(k - 1):
        g = f[:]
        for mask in range(1 << n):
            sub = mask
            while sub:
                v = max(f[mask ^ sub], s[sub])
                if v < g[mask]:
                    g[mask] = v
                sub = (sub - 1) & mask
        f = g
    return f[(1 << n) - 1]

print("1723 ví dụ [3,2,3], k=3 ->", min_max_load_k_workers([3, 2, 3], 3))
print("1723 ví dụ [1,2,4,7,8], k=2 ->", min_max_load_k_workers([1, 2, 4, 7, 8], 2))

1723 ví dụ [3,2,3], k=3 -> 3
1723 ví dụ [1,2,4,7,8], k=2 -> 11


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi phân hoạch | `O(B(n) n)` | `O(n)` | kiểm thử `n ≤ 8` |
| 2. Quay lui + cắt tỉa | sắp giảm, đối xứng, chặn | mũ | `O(n)` | phỏng vấn, chạy nhanh thực tế |
| 3. DP tập con | phiên cuối = submask | `O(3ⁿ)` | `O(2ⁿ)` | tổng quát (hàm theo nhóm) |
| 4. DP cặp | `(cnt, load)` từ điển | `O(n 2ⁿ)` | `O(2ⁿ)` | **nộp bài** |
| 5. Thử dần `k` | khả thi với `k` thùng | mũ | `O(n)` | khi đáp án sát cận dưới |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minSessions([2, 3, 3, 4, 4], 8)`?

**Bài 2 (tính tay):** Với `tasks = [1,2,3]`, `T = 3`, có bao nhiêu `mask` khác rỗng có tổng `≤ T`? Liệt kê.

**Bài 3 (điền TODO):** Hoàn thiện `min_sessions_cap(tasks, T, c)` — mỗi phiên làm **tối đa `c` việc**.

**Bài 4 (tự viết lại):** Viết lại DP cặp bằng **đệ quy có nhớ** `@lru_cache` trên `mask` (tải hiện tại suy ra được không? — gợi ý: trả về cặp).

**Bài 5 (mini challenge):** Ngoài số phiên tối thiểu, trả về **một cách chia cụ thể** (danh sách các phiên, mỗi phiên là danh sách chỉ số việc). Kiểm bằng validator: phủ đủ mỗi việc đúng một lần, mỗi phiên tổng `≤ T`, số phiên = tối ưu.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def min_sessions_cap(tasks, T, c):
    # TODO: DP tập con, chỉ nhận sub có s[sub] <= T và popcount(sub) <= c
    pass

# Bài 4 — TODO
def min_sessions_memo(tasks, T):
    # TODO: @lru_cache def go(mask) -> (cnt, load) nhỏ nhất theo từ điển
    pass

# Bài 5 — TODO
def min_sessions_with_plan(tasks, T):
    # TODO: DP tập con + lưu sub tối ưu cho mỗi mask, truy vết
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Tổng 16 = 2 × 8 ⇒ cận dưới 2.
- Mức 2: Tìm hai nhóm tổng 8.
- Mức 3: `{4,4}` và `{2,3,3}` ⇒ 2.

**Bài 2**
- Mức 1: Có 7 mask khác rỗng.
- Mức 2: Loại các tổng > 3: `{1,3}`=4, `{2,3}`=5, `{1,2,3}`=6.
- Mức 3: Còn 4 mask: `{1}`, `{2}`, `{3}`, `{1,2}`.

**Bài 3**
- Mức 1: Điều kiện thêm ở phiên cuối.
- Mức 2: `bin(sub).count('1') <= c`.
- Mức 3: Đối chiếu vét cạn có giới hạn số việc.

**Bài 4**
- Mức 1: `go(mask)` = cặp tốt nhất sau khi đã xếp `mask`.
- Mức 2: `go(0) = (1, 0)`; `go(mask) = min` trên bit `i` thuộc `mask` của bước chuyển từ `go(mask ^ (1<<i))`.
- Mức 3: Trả `go(full)[0]`.

**Bài 5**
- Mức 1: Lưu `choice[mask] = sub` tối ưu.
- Mức 2: Truy vết `mask → mask ^ choice[mask]` đến 0.
- Mức 3: Mỗi `sub` giải mã thành danh sách chỉ số.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
r1 = Solution().minSessions([2, 3, 3, 4, 4], 8)
print("Bài 1:", r1); assert r1 == 2 == min_sessions_brute([2, 3, 3, 4, 4], 8)

# Bài 2
tasks_, T_ = [1, 2, 3], 3
ok = [m for m in range(1, 8) if sum(tasks_[i] for i in range(3) if m >> i & 1) <= T_]
print("Bài 2:", len(ok), [[tasks_[i] for i in range(3) if m >> i & 1] for m in ok]); assert len(ok) == 4

# Bài 3
def min_sessions_cap_sol(tasks, T, c):
    n = len(tasks); s = subset_sums(tasks)
    f = [n + 1] * (1 << n); f[0] = 0
    for mask in range(1, 1 << n):
        sub = mask
        while sub:
            if s[sub] <= T and bin(sub).count('1') <= c:
                f[mask] = min(f[mask], f[mask ^ sub] + 1)
            sub = (sub - 1) & mask
    return f[-1]

def brute_cap(tasks, T, c):
    best = [len(tasks)]; loads = []; cnts = []
    def go(i):
        if i == len(tasks):
            best[0] = min(best[0], len(loads)); return
        for j in range(len(loads)):
            if loads[j] + tasks[i] <= T and cnts[j] < c:
                loads[j] += tasks[i]; cnts[j] += 1; go(i + 1); loads[j] -= tasks[i]; cnts[j] -= 1
        loads.append(tasks[i]); cnts.append(1); go(i + 1); loads.pop(); cnts.pop()
    go(0); return best[0]

rnd = random.Random(3)
for _ in range(200):
    n = rnd.randint(1, 7); T = rnd.randint(1, 15); c = rnd.randint(1, 4)
    a = [rnd.randint(1, min(10, T)) for _ in range(n)]
    assert min_sessions_cap_sol(a, T, c) == brute_cap(a, T, c)
print("Bài 3 OK, ví dụ [1,1,1,1], T=15, c=2 ->", min_sessions_cap_sol([1, 1, 1, 1], 15, 2))

Bài 1: 2
Bài 2: 4 [[1], [2], [1, 2], [3]]
Bài 3 OK, ví dụ [1,1,1,1], T=15, c=2 -> 2


In [27]:
# Bài 4
def min_sessions_memo_sol(tasks, T):
    n = len(tasks)
    @lru_cache(maxsize=None)
    def go(mask):
        if mask == 0:
            return (1, 0)
        best = (n + 1, 0)
        for i in range(n):
            if mask >> i & 1:
                cnt, load = go(mask ^ (1 << i))
                t = tasks[i]
                cand = (cnt, load + t) if load + t <= T else (cnt + 1, t)
                best = min(best, cand)
        return best
    return go((1 << n) - 1)[0]

# Bài 5
def min_sessions_with_plan_sol(tasks, T):
    n = len(tasks); s = subset_sums(tasks)
    f = [n + 1] * (1 << n); f[0] = 0; choice = [0] * (1 << n)
    for mask in range(1, 1 << n):
        sub = mask
        while sub:
            if s[sub] <= T and f[mask ^ sub] + 1 < f[mask]:
                f[mask] = f[mask ^ sub] + 1; choice[mask] = sub
            sub = (sub - 1) & mask
    plan, m = [], (1 << n) - 1
    while m:
        sub = choice[m]
        plan.append([i for i in range(n) if sub >> i & 1]); m ^= sub
    return f[-1], plan

def valid_plan(tasks, T, k, plan):
    idx = sorted(i for p in plan for i in p)
    return idx == list(range(len(tasks))) and all(sum(tasks[i] for i in p) <= T for p in plan) and len(plan) == k

rnd = random.Random(5)
for _ in range(300):
    n = rnd.randint(1, 9); T = rnd.randint(1, 15)
    a = [rnd.randint(1, min(10, T)) for _ in range(n)]
    opt = min_sessions_pair(a, T)
    assert min_sessions_memo_sol(a, T) == opt
    k, plan = min_sessions_with_plan_sol(a, T)
    assert k == opt and valid_plan(a, T, k, plan)
print("Bài 4, Bài 5 OK; ví dụ kế hoạch cho [5,3,6,5,7,4], T=15:", min_sessions_with_plan_sol([5, 3, 6, 5, 7, 4], 15))

Bài 4, Bài 5 OK; ví dụ kế hoạch cho [5,3,6,5,7,4], T=15: (2, [[2, 3, 5], [0, 1, 4]])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DP cặp (ít thùng nhất, sức chứa `T`):**

```python
dp = [(n + 1, 0)] * (1 << n); dp[0] = (1, 0)
for mask in range(1 << n):
    cnt, load = dp[mask]
    for i in range(n):
        if mask >> i & 1: continue
        t = w[i]
        cand = (cnt, load + t) if load + t <= T else (cnt + 1, t)
        dp[mask | 1 << i] = min(dp[mask | 1 << i], cand)
return dp[-1][0]
```

**Template DP tập con:** `f[mask] = min(f[mask ^ sub] + 1 for sub ⊆ mask if s[sub] <= T)`, duyệt `sub = (sub - 1) & mask`.

**Checklist:**
- [ ] Không dùng tham lam (FFD / next-fit)?
- [ ] Phiên mới có tải `t`, không phải 0?
- [ ] So sánh cả cặp `(cnt, load)`?
- [ ] Khởi tạo `dp[0] = (1, 0)` (một phiên đang mở)?
- [ ] `n = 14`: dùng `O(n·2ⁿ)` cho an toàn thời gian Python?
- [ ] Chữ ký đúng: `def minSessions(self, tasks: List[int], sessionTime: int) -> int`?